# Lab 04 · OpenMP Pitfalls · races, false sharing, NUMA, scheduling

Lab 03 got the pragma on. This lab covers the four ways your OpenMP program can be wrong or slow: **data races**, **false sharing**, **NUMA placement**, and **loop scheduling**. Each pitfall gets a runnable demo showing the bug and the fix.

**Prerequisites.** Lab 03 (working `heat2Domp.c`).

**Builds toward.** Lab 07 (hybrid MPI+OpenMP) inherits all of these — get them right here.

> **📚 Where to look when you're stuck**
>
> - [**OpenMP data-sharing attribute clauses**](https://www.openmp.org/spec-html/5.2/openmpsu53.html)
> - [**Intel: What is false sharing?**](https://herbsutter.com/2009/05/15/effective-concurrency-eliminate-false-sharing/)
> - [**NUMA and first-touch**](https://web.archive.org/web/2022/https://queue.acm.org/detail.cfm?id=2513149) (ACM Queue)



## How this notebook works

Same three surfaces as lab 01 and 02: **[Hub]** for orchestration, **[Hub -> Crux]** for ssh calls, and **[Crux compute]** inside the PBS script. Every code cell begins with `# [Where]`.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab04", host="crux",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + previous lab's artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab04 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
    check("lab03 openmp binary", remoteFileExists(env['HPC_LAB_DIR'].replace('lab04','lab03') + '/heat2Domp')),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> Crux] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab04 dir ready')


## Part 1 · Data race · what the sanitizer catches

A **data race** happens when two threads read+write the same memory location with at least one write, and no ordering enforced. The result is undefined — sometimes right, sometimes wrong, timing-dependent.

The classic OpenMP mistake: putting `#pragma omp parallel for` on a loop that accumulates into a shared scalar. Threads race on the write. The fix is a `reduction(+:sum)` clause.


In [ ]:
# [Hub] Write a demo: buggy parallel sum vs fixed reduction.
(labDir/'raceDemo.c').write_text('''
#include <stdio.h>
#include <omp.h>
int main(void) {
    long N = 100000000;
    double sumBuggy = 0.0, sumFixed = 0.0;
    #pragma omp parallel for
    for (long i = 0; i < N; i++) sumBuggy += 1.0;   /* RACE */
    #pragma omp parallel for reduction(+:sumFixed)
    for (long i = 0; i < N; i++) sumFixed += 1.0;   /* correct */
    printf("expected %ld,  buggy %.0f  fixed %.0f\\n", N, sumBuggy, sumFixed);
    return 0;
}
''')
showFile(labDir/'raceDemo.c', language='c', title='raceDemo.c')


In [ ]:
# [Hub -> Crux] Build with ThreadSanitizer + run. TSan catches the race.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cc -g -O1 -fopenmp             -o raceRaw   raceDemo.c
cc -g -O1 -fopenmp -fsanitize=thread -o raceSan raceDemo.c
echo === without TSan ===
OMP_NUM_THREADS=8 ./raceRaw
echo === with TSan ===
OMP_NUM_THREADS=8 ./raceSan 2>&1 | head -30
'''
pbsPath = labDir/'raceJob.pbs'
pbsPath.write_text(pbsHeader(name='lab04Race', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'), walltime='00:10:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/race.out') + jobBody)
sshPut(str(labDir/'raceDemo.c'), env['HPC_LAB_DIR']+'/raceDemo.c')
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/raceJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/raceJob.pbs'); waitJob(jobID, 15, 900)
sshGet(env['HPC_LAB_DIR']+'/race.out', str(labDir/'race.out'))
print((labDir/'race.out').read_text())


In [ ]:
checkpoint("Part 1 - data race demo", [
    check("race demo output", fileExists(str(labDir/'race.out'))),
])


## Part 2 · False sharing · when memory layout kills scaling

A cache line is typically **64 bytes**. When two threads write to two different variables that happen to live in the same cache line, the cache coherence protocol invalidates one thread's copy every time the other writes. The result is a mystery slowdown that gets worse with more threads.

Classic demo: an array of counters, one per thread, packed tight. Fix: pad each counter to its own cache line.


In [ ]:
# [Hub] Write the false-sharing demo.
(labDir/'fsDemo.c').write_text('''
#include <stdio.h>
#include <omp.h>
#include <time.h>
static double wall(void){struct timespec t;clock_gettime(CLOCK_MONOTONIC,&t);return t.tv_sec+t.tv_nsec*1e-9;}
int main(void){
    long iters = 100000000;
    long tight[16];  /* 16 longs = 128B, several threads share a cache line */
    long padded[16][8];  /* 8 longs per counter = 64B pad => own cache line each */
    for(int i=0;i<16;i++){tight[i]=0; padded[i][0]=0;}
    double t0=wall();
    #pragma omp parallel
    { int me=omp_get_thread_num(); for(long i=0;i<iters;i++) tight[me]++; }
    double tTight=wall()-t0;
    t0=wall();
    #pragma omp parallel
    { int me=omp_get_thread_num(); for(long i=0;i<iters;i++) padded[me][0]++; }
    double tPad=wall()-t0;
    printf("tight %.3fs  padded %.3fs  slowdown %.1fx\\n", tTight, tPad, tTight/tPad);
    return 0;
}
''')


In [ ]:
# [Hub -> Crux] Run at 8 threads. Expect padded to be several x faster.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cc -O2 -fopenmp -o fsDemo fsDemo.c
OMP_NUM_THREADS=8 OMP_PROC_BIND=close OMP_PLACES=cores ./fsDemo
'''
pbsPath = labDir/'fsJob.pbs'
pbsPath.write_text(pbsHeader(name='lab04FS', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'), walltime='00:10:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/fs.out') + jobBody)
sshPut(str(labDir/'fsDemo.c'), env['HPC_LAB_DIR']+'/fsDemo.c')
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/fsJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/fsJob.pbs'); waitJob(jobID, 15, 900)
sshGet(env['HPC_LAB_DIR']+'/fs.out', str(labDir/'fs.out'))
print((labDir/'fs.out').read_text())


In [ ]:
checkpoint("Part 2 - false sharing", [
    check("fs demo ran", fileExists(str(labDir/'fs.out'))),
])


## Part 3 · NUMA and first-touch

A Crux compute node has **two sockets**. Each socket has its own memory controllers; accessing memory local to your socket is fast, accessing memory on the other socket costs ~2x. The **first-touch** policy on Linux places each page on the socket of the thread that first writes it.

This means the initialization loop matters. If a single thread initializes the whole grid, the whole grid lives on that thread's socket, and half your threads pay remote-memory cost forever. Fix: **parallelize the init loop the same way as the compute loop**.


In [ ]:
# [Hub] Show the diff: init loop needs the same pragma.
print('Before fix: sequential init:')
print('    for (int i=0; i<N; i++) for (int j=0; j<N; j++) u[i*N+j] = ic(i,j);')
print()
print('After fix: parallel init - each thread first-touches its own strip:')
print('    #pragma omp parallel for schedule(static)')
print('    for (int i=0; i<N; i++) for (int j=0; j<N; j++) u[i*N+j] = ic(i,j);')
print()
print('Verify with: numactl --hardware; grep MemTotal /proc/self/numa_maps')


In [ ]:
# [Hub -> Crux] Confirm the node is 2-socket, then measure a heat2D run with
# vs without proper first-touch. This exercise leaves the source edit to you
# as an actual homework step - open heat2Domp.c and add the pragma to the init loop.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
echo === numa layout ===
numactl --hardware | head -20
'''
pbsPath = labDir/'numaJob.pbs'
pbsPath.write_text(pbsHeader(name='lab04Numa', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'), walltime='00:05:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/numa.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/numaJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/numaJob.pbs'); waitJob(jobID, 15, 600)
sshGet(env['HPC_LAB_DIR']+'/numa.out', str(labDir/'numa.out'))
print((labDir/'numa.out').read_text())


In [ ]:
checkpoint("Part 3 - NUMA layout inspected", [
    check("numactl output captured", fileExists(str(labDir/'numa.out'))),
])


## Part 4 · Loop scheduling · `static` vs `dynamic` vs `guided`

`schedule(static)` — divide iterations into equal chunks, one per thread. Zero runtime overhead. Right when iterations are equal cost.

`schedule(dynamic)` — thread grabs the next chunk when it finishes. Balances uneven workloads at the cost of runtime coordination.

`schedule(guided)` — dynamic, but chunk size shrinks over time. Compromise.

For the heat stencil, every iteration is exactly the same work, so **static** wins. For codes with load imbalance (adaptive meshes, particle work), dynamic usually wins even with the coordination overhead.


In [ ]:
# [Hub -> Crux] Sweep the three schedules on our stencil at 32 threads.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cp ../lab03/heat2Domp .
for s in static dynamic guided; do
  OMP_NUM_THREADS=32 OMP_SCHEDULE=$s \\
    ./heat2Domp --N 1024 --steps 200 --snapEvery 0 --outDir ./out --variant omp-$s
done
'''
pbsPath = labDir/'schedJob.pbs'
pbsPath.write_text(pbsHeader(name='lab04Sched', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'), walltime='00:15:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/sched.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/schedJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/schedJob.pbs'); waitJob(jobID, 20, 1200)
sshGet(env['HPC_LAB_DIR']+'/sched.out', str(labDir/'sched.out'))
sshGet(env['HPC_LAB_DIR']+'/out/timings.csv', str(labDir/'timings.csv'))
print((labDir/'sched.out').read_text()[-400:])


In [ ]:
# [Hub] Compare wall time across schedules.
import pandas as pd
df = pd.read_csv(labDir/'timings.csv')
schedDf = df[df['variant'].str.startswith('omp-')].sort_values('variant')
print(schedDf[['variant','threads','wall_s','mlups']].to_string(index=False))


In [ ]:
checkpoint("Part 4 - schedule sweep", [
    check("sched output", fileExists(str(labDir/'sched.out'))),
])


## Part 5 · The OMP environment variables you'll set for every run

| Variable | What it does | Sensible default |
|---|---|---|
| `OMP_NUM_THREADS` | number of threads | one per physical core |
| `OMP_PROC_BIND` | pin threads to cores? | `close` (pack) or `spread` |
| `OMP_PLACES` | what units to pin to | `cores` on modern CPUs |
| `OMP_SCHEDULE` | default `schedule(runtime)` value | `static` for regular work |
| `OMP_STACKSIZE` | per-thread stack | `16M` if you allocate on stack |

For any measurement run in this course, always set at least `OMP_NUM_THREADS`, `OMP_PROC_BIND=close`, and `OMP_PLACES=cores`. Without binding, the OS can migrate your threads and destroy locality mid-run.


In [ ]:
# [Hub] Print the recommended env-var line for cut+paste into your job scripts.
print('Standard OMP prologue for every compute-node job in this course:')
print()
print('export OMP_NUM_THREADS=$NCPUS')
print('export OMP_PROC_BIND=close')
print('export OMP_PLACES=cores')
print('export OMP_SCHEDULE=static')


In [ ]:
checkpoint("Part 5 - OMP env recipe recorded", [
    check("labEnv.sh exists", fileExists(str(labDir/'labEnv.sh'))),
])


## Part 6 · Bridge to lab 05

OpenMP scales to one node. Beyond that, you need **MPI** — the message-passing model that lets your program span many nodes.

Lab 05 is the MPI primer: point-to-point sends and receives, collectives, and the mental model of ranks talking to each other. Lab 06 applies MPI to the heat stencil with a proper 2D domain decomposition (the halo exchange).


## Wrap up

Moved the spine forward one lab. Ready for the next.


### Lab scorecard


In [ ]:
labSummary("OpenMP Pitfalls")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("OpenMP Pitfalls")
